In [23]:
import os
import textwrap
import math
import json
import pandas as pd
import numpy as np
from collections import Counter
from datetime import datetime
from dateutil.relativedelta import relativedelta
from IPython.display import display
from IPython.display import Markdown

from langchain_openai import ChatOpenAI

import api


def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatOpenAI(
  openai_api_key = os.getenv('OPENAI_API_KEY'),
  model='gpt-4o-mini',
  temperature=1,
)

# 買進賣出需要的手續費
fee = {
  'tax_stock' : 0.003,          # 買進與賣出各0.003
  'tax_future' : 0.00002 * 2,   # 買進與賣出各0.00002
  'fee_stock' : 0.001425 * 2,   # 券商手續費公道價，買進與賣出各0.001425
  'fee_future' : 0.00002 * 2,   # 個家不大相同，以交易稅計算
  'sliding_price' : 0.00001,    # 滑價
}

env = {
    "stock_id" : "TSLA",
    "country" : "us",
    "start_date" : "20231001",
    "end_date" : "20240930",
    "path_out" : "out_stock/GA_sot_4omini/",
}

# 載入 factors

In [24]:
factors = {}
path_factors = f"{env['path_out']}{env['stock_id']}/factors.json"
print(path_factors)
with open(path_factors, "r") as f:
  factors = json.load(f)

# Access by index using list of keys
key_list = list(factors.keys())
value_list = list(factors.values())
print(key_list)
print(len(key_list))
print(len(value_list))

out_stock/GA_sot_4omini/TSLA/factors.json
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30']
30
30


# 延伸factors

In [25]:
from api import factor_expanding

# 讀取舊資料
old_data = {}
path_expand = f"{env['path_out']}{env['stock_id']}/expand.json"
os.makedirs(os.path.dirname(path_expand), exist_ok=True)
if os.path.exists(path_expand):
    with open(path_expand, 'r') as f:
        old_data = json.load(f)

data = factor_expanding(llm, env, key_list, value_list, old_data)

# 儲存資料
with open(path_expand, 'w') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

running... factor_expanding
Already processed for date 20240930, skip this date
Already processed for date 20240927, skip this date
Already processed for date 20240926, skip this date
Already processed for date 20240925, skip this date
Already processed for date 20240924, skip this date
Already processed for date 20240923, skip this date
Already processed for date 20240920, skip this date
Already processed for date 20240919, skip this date
Already processed for date 20240918, skip this date
Already processed for date 20240917, skip this date
Already processed for date 20240916, skip this date
Already processed for date 20240913, skip this date
Already processed for date 20240912, skip this date
Already processed for date 20240911, skip this date
Already processed for date 20240910, skip this date
Already processed for date 20240909, skip this date
Already processed for date 20240906, skip this date
Already processed for date 20240905, skip this date
Already processed for date 20240904,

# GA 設定

In [26]:
import json
import random
from datetime import datetime
from datetime import timedelta

def split_expand(env, path_expand):
    # 解析日期
    start_day = datetime.strptime(env['start_date'], '%Y%m%d')
    end_day = datetime.strptime(env['end_date'], '%Y%m%d')

    # read train data
    with open(path_expand, 'r') as f:
        content = f.read().strip()
        data_points = json.loads(content)

    # split to training and testing
    total_days = (end_day - start_day).days + 1
    split_point = int(total_days * 3 / 4)
    training_end_day = start_day + timedelta(days=split_point - 1)
    testing_start_day = training_end_day + timedelta(days=1)
    train_datarange = {}
    for date_str, value in data_points["output_data"].items():
        date = datetime.strptime(date_str, '%Y%m%d')
        if start_day <= date <= training_end_day:
            train_datarange[date_str] = value
    test_datarange = {}
    for date_str, value in data_points["output_data"].items():
        date = datetime.strptime(date_str, '%Y%m%d')
        if testing_start_day <= date <= end_day:
            test_datarange[date_str] = value



    return train_datarange, test_datarange

In [27]:
train_datarange, test_datarange = split_expand(env, path_expand)
print(len(train_datarange))
print(train_datarange.keys())
print(len(test_datarange))
print(test_datarange.keys())


187
dict_keys(['20240628', '20240627', '20240626', '20240625', '20240624', '20240621', '20240620', '20240618', '20240617', '20240614', '20240613', '20240612', '20240611', '20240610', '20240607', '20240606', '20240605', '20240604', '20240603', '20240531', '20240530', '20240529', '20240528', '20240524', '20240523', '20240522', '20240521', '20240520', '20240517', '20240516', '20240515', '20240514', '20240513', '20240510', '20240509', '20240508', '20240507', '20240506', '20240503', '20240502', '20240501', '20240430', '20240429', '20240426', '20240425', '20240424', '20240423', '20240422', '20240419', '20240418', '20240417', '20240416', '20240415', '20240412', '20240411', '20240410', '20240409', '20240408', '20240405', '20240404', '20240403', '20240402', '20240401', '20240328', '20240327', '20240326', '20240325', '20240322', '20240321', '20240320', '20240319', '20240318', '20240315', '20240314', '20240313', '20240312', '20240311', '20240308', '20240307', '20240306', '20240305', '20240304', '

# GA Training and Testing

Accuracy

In [28]:
from api import genetic_algorithm
from api import eval

# setting --------------------------------
mode = 0     # 0: 準確率, 1: 期望值
# ----------------------------------------

path_str = "exp"
if mode == 0:
    path_str = path_str + "_ac"
else:
    path_str = path_str + "_ev"

out_folder = f"{env['path_out']}/{env['stock_id']}/{env['start_date']}_{env['end_date']}" + f"{path_str}"
state_file= out_folder + '/state.json'
file_gen= out_folder + '/generation_results.json'
file_result= out_folder + '/result.json'


print("Start, 第一次跑的話請確認state是空的")
best_individual = genetic_algorithm(factors, population_size=20, generations=50, state_file=state_file, results_file=file_gen, env=env, mode=mode, datarange=train_datarange)
print(f"Best individual: {best_individual}")

individual = best_individual
res_train = eval(individual, env['stock_id'], train_datarange, env)
res_test = eval(individual, env['stock_id'], test_datarange, env)
res = {
    "train": res_train,
    "test": res_test,
    "individual": individual,
}
print(res)
print(file_result)
with open(file_result, 'w') as f:
    json.dump(res, f, ensure_ascii=False, indent=4)

Start, 第一次跑的話請確認state是空的
Best individual: [0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0]
{'train': {'tp': 17, 'fp': 13, 'tn': 83, 'fn': 74, 'avail_count': 187, 'ttl_count': 187, 'accuracy': 0.5347593582887701, 'precision': 0.5666666666666667, 'recall': 0.18681318681318682, 'ev': 0.0010359104036120198, 'precision_ev': 0.0030375634098591744}, 'test': {'tp': 13, 'fp': 4, 'tn': 24, 'fn': 23, 'avail_count': 64, 'ttl_count': 64, 'accuracy': 0.578125, 'precision': 0.7647058823529411, 'recall': 0.3611111111111111, 'ev': 0.002605639790637255, 'precision_ev': 0.009800562033183965}, 'individual': [0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0]}
out_stock/GA_sot_4omini//TSLA/20231001_20240930exp_ac/result.json


In [29]:
df = pd.DataFrame([res['train'], res['test']], index=['train', 'test'])
df

,tp,fp,tn,fn,avail_count,ttl_count,accuracy,precision,recall,ev,precision_ev
train,17,13,83,74,187,187,0.534759,0.566667,0.186813,0.001036,0.003038
test,13,4,24,23,64,64,0.578125,0.764706,0.361111,0.002606,0.009801


Expected Value

In [30]:
from api import genetic_algorithm
from api import eval

# setting --------------------------------
mode = 1     # 0: 準確率, 1: 期望值
# ----------------------------------------

path_str = "exp"
if mode == 0:
    path_str = path_str + "_ac"
else:
    path_str = path_str + "_ev"

out_folder = f"{env['path_out']}/{env['stock_id']}/{env['start_date']}_{env['end_date']}" + f"{path_str}"
state_file= out_folder + '/state.json'
file_gen= out_folder + '/generation_results.json'
file_result= out_folder + '/result.json'


print("Start, 第一次跑的話請確認state是空的")
best_individual = genetic_algorithm(factors, population_size=20, generations=50, state_file=state_file, results_file=file_gen, env=env, mode=mode, datarange=train_datarange)
print(f"Best individual: {best_individual}")

individual = best_individual
res_train = eval(individual, env['stock_id'], train_datarange, env)
res_test = eval(individual, env['stock_id'], test_datarange, env)
res = {
    "train": res_train,
    "test": res_test,
    "individual": individual,
}
print(res)
print(file_result)
with open(file_result, 'w') as f:
    json.dump(res, f, ensure_ascii=False, indent=4)

Start, 第一次跑的話請確認state是空的
Best individual: [1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1]
{'train': {'tp': 18, 'fp': 15, 'tn': 80, 'fn': 73, 'avail_count': 186, 'ttl_count': 187, 'accuracy': 0.5268817204301075, 'precision': 0.5454545454545454, 'recall': 0.1978021978021978, 'ev': 0.0009772737570117058, 'precision_ev': 0.0026552355837656354}, 'test': {'tp': 13, 'fp': 4, 'tn': 24, 'fn': 23, 'avail_count': 64, 'ttl_count': 64, 'accuracy': 0.578125, 'precision': 0.7647058823529411, 'recall': 0.3611111111111111, 'ev': 0.002605639790637255, 'precision_ev': 0.009800562033183965}, 'individual': [1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1]}
out_stock/GA_sot_4omini//TSLA/20231001_20240930exp_ev/result.json


In [31]:
df = pd.DataFrame([res['train'], res['test']], index=['train', 'test'])
df

,tp,fp,tn,fn,avail_count,ttl_count,accuracy,precision,recall,ev,precision_ev
train,18,15,80,73,186,187,0.526882,0.545455,0.197802,0.000977,0.002655
test,13,4,24,23,64,64,0.578125,0.764706,0.361111,0.002606,0.009801


# show old data

In [32]:
import os
import json
import pandas as pd
from api import eval

folders = os.listdir(env['path_out'])
results = []
index = []

for folder in folders:
    if folder == "factors_eng.json" or folder == "factors.json" or folder == "old_expand":
        continue
    folders_result = os.listdir(f"{env['path_out']}{folder}")
    # print(f"{folder}:")
    for file in folders_result:
        if file == "factors.json" or file == "factors1.json" or file == "factors2.json" or file == "expand.json" or file == "expand1.json" or file == "expand2.json":
            continue
        path = f"{env['path_out']}{folder}/{file}/result.json"
        with open(path, 'r') as f:
            json_data = json.load(f)
        # print(json_data)
        results.append(json_data['test'])
        index.append(f"{folder}_{file}")
        

print(f"共有 {len(results), len(index)} 筆資料")
print(results)

df = pd.DataFrame(results, index=index)
selected_columns = ['accuracy', 'precision', 'ev', 'precision_ev']  # 替換為您需要的欄位名稱
df_selected = df.loc[:, selected_columns]
df_sorted = df_selected.sort_index()
df_sorted

共有 (16, 16) 筆資料
[{'tp': 19, 'fp': 18, 'tn': 4, 'fn': 6, 'avail_count': 47, 'ttl_count': 47, 'accuracy': 0.48936170212765956, 'precision': 0.5135135135135135, 'recall': 0.76, 'ev': 1.8130582968558726e-06, 'precision_ev': 0.000490741739743881}, {'tp': 19, 'fp': 18, 'tn': 4, 'fn': 6, 'avail_count': 47, 'ttl_count': 47, 'accuracy': 0.48936170212765956, 'precision': 0.5135135135135135, 'recall': 0.76, 'ev': 1.8130582968558726e-06, 'precision_ev': 0.000490741739743881}, {'tp': 17, 'fp': 28, 'tn': 9, 'fn': 9, 'avail_count': 63, 'ttl_count': 64, 'accuracy': 0.4126984126984127, 'precision': 0.37777777777777777, 'recall': 0.6538461538461539, 'ev': -0.0010705071865823052, 'precision_ev': -0.0019024197245643294}, {'tp': 17, 'fp': 25, 'tn': 11, 'fn': 10, 'avail_count': 63, 'ttl_count': 64, 'accuracy': 0.4444444444444444, 'precision': 0.40476190476190477, 'recall': 0.6296296296296297, 'ev': -0.001617349351489699, 'precision_ev': -0.0024345897540011333}, {'tp': 24, 'fp': 15, 'tn': 11, 'fn': 14, 'avai

,accuracy,precision,ev,precision_ev
2330_20231001_20240930exp_ac,0.452381,0.400000,0.002779,0.001736
2330_20231001_20240930exp_ev,0.547170,0.511628,0.003996,0.002774
2454_20231001_20240930exp_ac,0.480769,0.400000,0.003241,0.003396
2454_20231001_20240930exp_ev,0.557692,0.461538,0.004684,0.005279
AAPL_20231001_20240930exp_ac,0.546875,0.615385,-0.000777,0.000695
AAPL_20231001_20240930exp_ev,0.546875,0.615385,-0.000777,0.000695
AMZN_20231001_20240630exp_ac,0.489362,0.513514,0.000002,0.000491
AMZN_20231001_20240630exp_ev,0.489362,0.513514,0.000002,0.000491
AMZN_20231001_20240930exp_ac,0.412698,0.377778,-0.001071,-0.001902
AMZN_20231001_20240930exp_ev,0.444444,0.404762,-0.001617,-0.002435


In [33]:
# import os
# import json
# import pandas as pd
# from api import eval

# folders = os.listdir(env['path_out'])
# results = []
# index = []
# print(env['start_date'], env['end_date'])

# for folder in folders:
#     if folder == "factors_eng.json" or folder == "factors.json" or folder == "old_expand":
#         continue
#     folders_result = os.listdir(f"{env['path_out']}{folder}")
#     print(f"{folder}:")
#     for file in folders_result:
#         if file == "factors.json" or file == "expand.json" or file == "expand1.json" or file == "expand2.json":
#             continue
#         path = f"{env['path_out']}{folder}/{file}/generation_results.json"
#         with open(path, 'r') as f:
#             content = f.read().strip()
#         # print(file)
#         best_individual = []
#         for text in content.split('\n'):
#             data = json.loads(text)
#             best_individual = data['best_individual']

#         try:
#             individual = best_individual
#             print(f"{env['path_out']}{folder}/expand.json")
#             train_datarange, test_datarange = split_expand(env, f"{env['path_out']}{folder}/expand.json")
#             res_train = eval(individual, env['stock_id'], train_datarange)
#             res_test = eval(individual, env['stock_id'], test_datarange)
#             res = {
#                 "train": res_train,
#                 "test": res_test,
#                 "individual": individual,
#             }

#             file_result = f"{env['path_out']}{folder}/{file}/result.json"
#             print(file_result)
#             with open(file_result, 'w') as f:
#                 json.dump(res, f, ensure_ascii=False, indent=4)
#         except:
#             print("\tError")
#             pass


